# Lesson 10 — Agentic RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/10-agentic-rag/colab.ipynb)

*The pipeline stops being a straight line and starts making decisions — which means it can now fail in loops.*

**What you will learn**

- Why a similarity threshold structurally cannot catch some unanswerable questions
- Corrective RAG (CRAG): grade the retrieved context, then answer, widen, or refuse
- Why this repo implements CRAG but not Self-RAG reflection: fix a failure at the earliest stage that can see it
- Multi-hop retrieval: letting the model write the follow-up query
- Loop guards — hop caps, monotonic accumulation, convergence detection, failing open — demonstrated with fake models

| | |
|---|---|
| ⏱️ Time | 60–75 min |
| 🖥️ Runtime | **T4 GPU recommended** — grading costs a model call per question |
| 🤖 Needs the generator (Ollama) | yes — for some cells |

**Before this:** [09-routing-and-multi-index](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/09-routing-and-multi-index/colab.ipynb)

> **How to use this notebook.** Run the cells top to bottom (`Shift+Enter`). Every
> time you see **🔮 Predict**, stop and write down what you expect *before* running
> the next cell — the gap between prediction and result is where the learning is.
> Answers to the check-your-understanding questions are hidden behind ▶ toggles.

## 0. Setup

This cell clones the course repo into the Colab runtime, installs the
dependencies, and (if `USE_LLM` is ticked) installs and starts
[Ollama](https://ollama.com) with the small `qwen2.5:1.5b` generator. It is safe
to re-run. Everything in the repo's `ragkit/` library becomes importable.

If an import fails after the install, use **Runtime → Restart session** and run
this cell again.

In [ ]:
# @title ⚙️ Setup — run me first
USE_LLM = True  # @param {type:"boolean"}
BRANCH = "main"  # @param {type:"string"}

import subprocess
import sys
from pathlib import Path


def _find_repo():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "ragkit").is_dir() and (p / "colab" / "colab_setup.py").exists():
            return p
    return None


REPO = _find_repo()
if REPO is None:
    REPO = Path("/content/rag-from-scratch-to-production")
    if not REPO.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                BRANCH,
                "https://github.com/safar-byte/rag-from-scratch-to-production.git",
                str(REPO),
            ],
            check=True,
        )
sys.path.insert(0, str(REPO / "colab"))
from colab_setup import compare_runs, eval_frames, results_frame, setup  # noqa: F401

LLM_READY = setup(llm=USE_LLM)

## 1. Why this lesson exists: the other half of the refusal problem

Lesson 03 found the worst result in the repo — the pipeline invents answers to questions
the corpus does not cover. A strict prompt and a score floor (lesson 12) each help. The
score floor structurally cannot catch the rest, and the reason is worth seeing:

In [ ]:
import pandas as pd

from ragkit.generate import abstain
from ragkit.pipeline import RagPipeline

pipeline = RagPipeline(strategy="rerank")
if pipeline.count() == 0:
    print(pipeline.ingest())

pairs = [
    ("What is the default value of RETRIEVAL_FANOUT?", "real setting"),
    ("What is the default value of MAX_CHUNK_BYTES?", "INVENTED setting"),
    ("What is the default value of the SHARD_REPLICATION_FACTOR setting?", "INVENTED setting"),
    ("What port does the query service listen on by default?", "real"),
    ("What is the capital of France?", "off-topic"),
]
pd.DataFrame(
    [
        {
            "question": q,
            "truth": t,
            "best dense similarity": round(
                abstain.best_retrieval_score(pipeline.retrieve(q, top_k=5)), 3
            ),
        }
        for q, t in pairs
    ]
)

The invented settings score about as high as the real one — sometimes *higher*. They
are questions about configuration defaults over a corpus full of configuration
defaults, so they retrieve the same passages with the same confidence. **No threshold
separates them**, because a similarity score measures *topical closeness*, not
*answerability*.

Telling them apart requires **reading the passages** and asking whether they contain
the answer. That is a model call. That is CRAG.

## 2. Corrective RAG

```
retrieve ──► grade the passages
                ├── correct    ──► answer
                ├── ambiguous  ──► widen the search once, then answer
                └── incorrect  ──► refuse  (the generator is never called)
```

The grader's entire job is one distinction, which its prompt hammers on:

In [ ]:
from ragkit.agent.grade import GRADER_SYSTEM

print(GRADER_SYSTEM)

*Being about the same topic is not the same as containing the answer.* Most grading
mistakes are made by scoring topical similarity — which is exactly what the cheap
threshold already does, so a grader that makes that mistake adds cost and nothing else.

### Grade some questions

In [ ]:
from ragkit.agent import ContextGrader

if LLM_READY:
    grader = ContextGrader(pipeline.generator)
    rows = []
    for q, truth in pairs:
        g = grader.grade(q, pipeline.retrieve(q, top_k=5))
        rows.append(
            {
                "question": q[:55],
                "truth": truth,
                "verdict": g.verdict.value,
                "confidence": g.confidence,
                "reason": g.reason,
            }
        )
    display(pd.DataFrame(rows))

🔍 **Check the verdicts against the "truth" column.** A good grader says `incorrect` for
the invented settings and France, and `correct` for the real ones. When this notebook
was written, the 1.5B model graded **every** question `incorrect` — real ones included,
all at the same confidence. Keep that in mind for section 3.

### The full CRAG loop

`CorrectiveRag.ask` retrieves, grades, and then answers, widens or refuses. The
`incorrect` branch never calls the generator — the model is never given the
opportunity to invent something plausible from passages that do not support it.

In [ ]:
from ragkit.agent import CorrectiveRag

if LLM_READY:
    crag = CorrectiveRag(pipeline.retriever, pipeline.generator, ContextGrader(pipeline.generator))
    for q in [
        "What is the default value of the SHARD_REPLICATION_FACTOR setting?",
        "What does exit status 75 from the indexer mean?",
    ]:
        answer = crag.ask(q, top_k=5)
        t = answer.trace
        print(
            f"Q: {q}\n   verdict={t['crag_verdict']} ({t['crag_reason']}) · steps: {' → '.join(t['crag_steps'])}"
            f" · {t['total_ms']:.0f} ms\n   A: {answer.text}\n"
        )

## 3. Measure it — refusals *and* false refusals

CRAG has to beat the free baseline. Lesson 03 measured the **strict prompt** at 0.900
correct refusal for free, at the price of one false refusal in five. So the interesting
question (the repo's open item #2) is not "does CRAG refuse?" but **does it reach the
strict prompt's refusal rate with fewer false refusals — and is that worth a model
call?** Let's measure on all 10 unanswerable questions plus 10 answerable lookups.

In [ ]:
import time

from ragkit.eval.golden import load_golden
from ragkit.eval.judge import looks_like_refusal
from ragkit.generate import generate_answer

golden = load_golden()
subset = [q for q in golden if q.kind == "unanswerable"] + [q for q in golden if q.kind == "lookup"]

rows = []
if LLM_READY:
    for name in ["strict prompt", "CRAG (+ strict prompt)"]:
        started = time.perf_counter()
        for q in subset:
            if name == "strict prompt":
                a = generate_answer(
                    q.question,
                    pipeline.retrieve(q.question, top_k=5),
                    pipeline.generator,
                    strict=True,
                )
                refused = looks_like_refusal(a.text)
            else:
                a = crag.ask(q.question, top_k=5)
                refused = a.trace.get("abstained", False) or looks_like_refusal(a.text)
            rows.append(
                {
                    "config": name,
                    "kind": q.kind,
                    "question": q.question,
                    "refused": refused,
                    "verdict": a.trace.get("crag_verdict"),
                    "answer": a.text[:100],
                }
            )
        print(f"{name}: {time.perf_counter() - started:.0f}s")

    results = pd.DataFrame(rows)
    display(
        results.groupby("config").apply(
            lambda g: pd.Series(
                {
                    "correct refusal (10 unanswerable)": g[g.kind == "unanswerable"].refused.mean(),
                    "false refusals (10 answerable)": int(
                        g[g.kind != "unanswerable"].refused.sum()
                    ),
                }
            ),
            include_groups=False,
        )
    )

In [ ]:
if LLM_READY:
    display(
        results[results.config.str.startswith("CRAG")][["kind", "question", "verdict", "refused"]]
    )

**Read the verdicts, not just the rate.** When this notebook was written:

| | correct refusal (10 unanswerable) | false refusals (10 answerable) | model calls |
|---|---|---|---|
| strict prompt | 0.900 | 1 / 10 | 1 per question |
| CRAG with the 1.5B grader | **1.000** | **8 / 10** | 2 per question |

A perfect refusal rate — bought by refusing *almost everything*. The small grader marks
nearly every set of passages `incorrect`, including "What does exit status 75 mean?",
whose answer is sitting in the top passage. That is the failure the false-refusal
control exists to catch: **looked at alone, the refusal column says CRAG is the best
lever in the course.**

This answers the repo's open question for the local profile: with a 1.5B grader, CRAG
is strictly worse than the free strict prompt, at twice the model calls. Grading
answerability is a *harder* task than answering — the grader must judge what is
*missing* — so it needs a stronger model than the generator, not the same small one.
On the cloud profile (Claude Haiku as the grader) the result may well flip; it has not
been measured. Measure it before believing either outcome.

## 4. Why not Self-RAG reflection?

CRAG grades the *retrieval*. The natural next step is grading the *answer*: generate,
check it against the context, regenerate if ungrounded (Self-RAG). This repo deliberately
does not, because:

- Reflection **doubles or triples generation cost** on every query.
- On this corpus the measured groundedness problem is concentrated in the unanswerable
  questions — which CRAG (and the strict prompt) already catch *before* generation.

> **Fix a failure at the earliest stage that can see it.** Refusing before generation is
> cheaper and more reliable than generating and then catching it. Paying for a fix to a
> problem already fixed upstream is the most common way agentic pipelines get expensive.

## 5. Multi-hop retrieval

Some questions cannot be answered by any single retrieval, because the terms needed for
the second search appear only in the results of the first:

> *Dense retrieval fails on error codes. Which retrieval method fixes that, and why does
> it work where embeddings do not?*

Nothing in the question says "BM25" — that document is reachable only after reading the
first result. `MultiHopRetriever` lets the model write the follow-up query:

In [ ]:
from ragkit.agent import MultiHopRetriever
from ragkit.agent.multihop import FOLLOWUP_SYSTEM

print(FOLLOWUP_SYSTEM, "\n")
if LLM_READY:
    hopper = MultiHopRetriever(pipeline.retriever, pipeline.generator, max_hops=2)
    q = (
        "Dense retrieval fails on error codes. Which retrieval method fixes that, "
        "and why does it work where embeddings do not?"
    )
    hits = hopper.retrieve(q, top_k=5)
    trace = hopper.last_trace
    print("queries written:", *[f"\n   {i}. {x}" for i, x in enumerate(trace.queries)])
    print("hops:", trace.hops, "\nstopped because:", trace.stopped_because)
    display(results_frame(hits, scores=False))

Does it help on the golden set's multi-hop questions? (7 questions, ≤ 2 model calls each.)

In [ ]:
from ragkit.eval.metrics import recall_at_k

if LLM_READY:
    rows = []
    for q in [q for q in golden if q.kind == "multi_hop"]:
        one = list(dict.fromkeys(s.chunk.doc_id for s in pipeline.retrieve(q.question, top_k=5)))
        many = list(dict.fromkeys(s.chunk.doc_id for s in hopper.retrieve(q.question, top_k=5)))
        rows.append(
            {
                "question": q.question[:60],
                "relevant": q.relevant,
                "R@5 single": recall_at_k(one, q.relevant, 5),
                "R@5 multi-hop": recall_at_k(many, q.relevant, 5),
                "hops": len(hopper.last_trace.queries) - 1,
                "stopped": hopper.last_trace.stopped_because,
            }
        )
    mh = pd.DataFrame(rows)
    display(mh)
    print(
        "mean R@5 — single:",
        mh["R@5 single"].mean().round(3),
        "| multi-hop:",
        mh["R@5 multi-hop"].mean().round(3),
    )

Single-shot hybrid + rerank already finds most multi-hop documents on this small
corpus (lesson 03 showed multi_hop R@3 ≈ 0.93), so expect little or no gain — and note
the cost in model calls. Only 7 of 44 questions are multi-hop: running hops on every
query multiplies cost for nothing, which is an argument for **routing into** multi-hop
(lesson 09) rather than defaulting to it.

## 6. Loop guards — the part that keeps agents safe

A loop that decides its own inputs needs bounds it does not control. `MultiHopRetriever`
enforces three:

1. **A hard cap** — `max_hops` bounds extra searches.
2. **Monotonic accumulation** — results only grow (deduplicated by chunk id); a bad
   follow-up wastes time but cannot destroy a good result set.
3. **Convergence detection** — a follow-up that repeats an earlier query means the loop
   has converged; continuing would burn the whole budget for nothing.

We can prove each one **without any real model**, using scripted fake generators — the
same technique the test suite uses.

In [ ]:
from ragkit.types import Usage


class ScriptedGenerator:
    """Replies from a fixed script, then repeats the last line forever."""

    name = "scripted"

    def __init__(self, *replies):
        self.replies, self.calls = list(replies), 0

    def generate(self, *, system, prompt, max_tokens=1024):
        reply = self.replies[min(self.calls, len(self.replies) - 1)]
        self.calls += 1
        if isinstance(reply, Exception):
            raise reply
        return reply, Usage()


scenarios = {
    "model is satisfied": ScriptedGenerator("ENOUGH"),
    "model keeps rephrasing": ScriptedGenerator(
        "SEARCH: bm25 error codes", "SEARCH: BM25 error codes"
    ),
    "model never stops": ScriptedGenerator("SEARCH: q1", "SEARCH: q2", "SEARCH: q3", "SEARCH: q4"),
    "model crashes": ScriptedGenerator(RuntimeError("model server down")),
}
rows = []
for name, gen in scenarios.items():
    hopper_fake = MultiHopRetriever(pipeline.retriever, gen, max_hops=2)
    out = hopper_fake.retrieve("Which method fixes error-code lookup?", top_k=5)
    rows.append(
        {
            "scenario": name,
            "model calls": gen.calls,
            "searches": len(hopper_fake.last_trace.queries),
            "stopped because": hopper_fake.last_trace.stopped_because or "planner failed → stop",
            "results returned": len(out),
        }
    )
pd.DataFrame(rows)

Every scenario terminates, and even the crashing model still returns the first hop's
results.

🔍 **But look at the "model crashes" row's reason.** It says *"model said enough"*.
`_next_query` catches the exception and returns `None` — the same signal as a
deliberate `ENOUGH` — so the trace misreports *why* the loop stopped. The loop is safe;
the **trace lies**. In production that turns "the planner has been down all week" into
"the planner keeps deciding one hop is enough", which looks healthy. (Found while
writing this notebook — fixing it is exercise 4.)

### Failing open

The grader **fails open**: if it errors or returns unparseable output, the query
proceeds ungraded rather than failing. An unavailable grader should degrade the system
to its previous behaviour, not take it down. The cost of that choice: a silently broken
grader looks like a working one — so `Grading.reason` records what happened.

In [ ]:
for gen in [ScriptedGenerator(RuntimeError("timeout")), ScriptedGenerator("I think it's fine!")]:
    g = ContextGrader(gen).grade("anything", pipeline.retrieve("anything", top_k=3))
    print(f"verdict={g.verdict.value:8s} confidence={g.confidence}  reason={g.reason!r}")

Every bound in `ragkit/agent/` is a **constructor argument**, not a constant buried in a
branch — because the bound is a deployment decision, and a limit you cannot see is a
limit you will not tune.

## 7. Check your understanding

**1. Why can't a score floor catch "What is the default value of MAX_CHUNK_BYTES?"**

<details><summary>▶ Show answer</summary>

It retrieves the same configuration passages, with the same similarity, as a real
configuration question. Similarity measures topical closeness, not whether the
passages contain the answer. Only reading them (a grader) can tell.

</details>

**2. Why does CRAG's `incorrect` branch skip the generator entirely?**

<details><summary>▶ Show answer</summary>

So the model never gets the chance to produce a plausible invention from passages
that do not support it — and because it is cheaper. Refusing at the earliest stage
that can see the problem is the most reliable fix.

</details>

**3. The grader fails open. What is the risk, and how is it mitigated?**

<details><summary>▶ Show answer</summary>

A broken grader (down, or returning junk) silently passes everything, looking like a
working one. `Grading.reason` records "grader unavailable" / "unparseable output",
and an eval that reports the verdict distribution would expose a grader that never
says anything but "correct".

</details>

## ✍️ Exercises

1. **Break the loop guards deliberately.** Comment out the convergence check (copy the
   class) and re-run the "keeps rephrasing" scenario. How many model calls now?
2. **Make the grader fail** with a scripted reply like `{"verdict": "maybe"}`. What does
   `CorrectiveRag` do?
3. **Argue the other side.** Describe a corpus where Self-RAG reflection *would* earn its
   cost over CRAG.
4. **Make the trace honest.** Change `MultiHopRetriever` so a planner failure is
   recorded as `"planner failed: <error>"` rather than `"model said enough"`, and add a
   test with a raising fake generator.
5. **Give CRAG a fair chance.** On a GPU runtime, pull a stronger grader with
   `setup(model="qwen2.5:7b")`, build it with
   `get_generator(Settings(local_generation_model="qwen2.5:7b"))`, pass it to
   `ContextGrader`, and re-run section 3 (keep the small model as the answerer). Does
   the false-refusal count fall?

## Key takeaways

1. A similarity threshold measures topic, not answerability — some fakes always pass it.
2. CRAG reads the passages and refuses before generation — but with a 1.5B grader it refused 8 of 10 answerable questions; always measure false refusals.
3. Fix a failure at the earliest stage that can see it; that is why there is no Self-RAG here.
4. Multi-hop lets the model write follow-up queries — useful for a minority of questions, so route into it.
5. Agent loops need external bounds: hop caps, monotonic results, convergence checks, fail-open graders.

## Next

**[Lesson 11 — GraphRAG](https://colab.research.google.com/github/safar-byte/rag-from-scratch-to-production/blob/main/lessons/11-graphrag/colab.ipynb)**